In [87]:
import pandas as pd
import re
from collections import Counter

In [27]:
pd.set_option('display.max_colwidth', None)

df = pd.read_csv("train.csv")
df.tail()

,Code Switched,Arabic
19995,ويعتبر هذا الـ meeting أول لقاء إيراني أميركي رسمي خارج جلسات التفاوض الرسمية التي تشارك فيها القوى الست الكبرى.,ويعتبر هذا الاجتماع أول لقاء إيراني أميركي رسمي خارج جلسات التفاوض الرسمية التي تشارك فيها القوى الست الكبرى.
19996,وتعد سيدة في مجال الـ cars الرياضية ذات المحرك ثماني الأسطوانات، إذ تم إنتاج أكثر من مليون ونصف المليون نسخة.,وتعد سيدة في مجال السيارات الرياضية ذات المحرك ثماني الأسطوانات، إذ تم إنتاج أكثر من مليون ونصف المليون نسخة.
19997,وقالت وزيرة الإعلام اليمنية نادية السقاف إن مقاتلين حوثيين فتحوا النار على قافلة الـ prime minister خالد بحاح في صنعاء اليوم الاثنين لكن لم تقع اصابات.,وقالت وزيرة الإعلام اليمنية نادية السقاف إن مقاتلين حوثيين فتحوا النار على قافلة رئيس الوزراء خالد بحاح في صنعاء اليوم الاثنين لكن لم تقع اصابات.
19998,ذلك أن أكبر انخفاض لنسب الفقر قد سجل في دول الخليج، كما سجل تحسن في مؤشرات الصحة والـ education والبيئة.,ذلك أن أكبر انخفاض لنسب الفقر قد سجل في دول الخليج، كما سجل تحسن في مؤشرات الصحة والتعليم والبيئة.
19999,الصرع يعد من الـ diseases العصبية التي تصيب الكبار والصغار، ونوبات الصرع الشديدة تستعصي على الـ treatment ما يزيد من ألم ومعاناة المصابين.,الصرع يعد من الأمراض العصبية التي تصيب الكبار والصغار، ونوبات الصرع الشديدة تستعصي على العلاج ما يزيد من ألم ومعاناة المصابين.


## Text cleaning

In [28]:
df.isnull().sum()

Code Switched    0
Arabic           0
dtype: int64

In [29]:
df.duplicated().sum()

np.int64(0)

In [74]:
def check_df(df):
    for i in range(len(df)):
        for c in df.columns:
            text = df[c][i]

            # 1. Missing or empty sentences
            if pd.isna(text) or not str(text).strip():
                print(f"Row {i}, Column {c}: Missing/empty")
                continue

            # 2. Extra whitespace
            if re.search(r'\s{2,}', text) or text != text.strip():
                print(f"Row {i}, Column {c}: Extra whitespace")

            # 3. Consecutive duplicate words
            duplicates = re.findall(r'\b(\w+)\s+\1\b', text, flags=re.IGNORECASE)
            if duplicates:
                print(f"Row {i}, Column {c}: Repeated words {duplicates}")

            # 4. Language validation
            if c == "Code Switched":
                if not re.search(r'[ء-ي]', text) or not re.search(r'[A-Za-z]', text):
                    print(f"Row {i}: Input missing Arabic or English")

            elif c == "Arabic":
                if not re.search(r'[ء-ي]', text) or re.search(r'[A-Za-z]', text):
                    print(f"Row {i}: Invalid Arabic target")

In [75]:
check_df(df)

## Tokenization

In [80]:
token = re.compile(
    r'[ء-يٱ][ء-يٱً-ٰٟـ]*'
    r'|[A-Za-z]+'
    r'|\d+(?:[.,/]\d+)*'
    r'|[^\w\s]',
    re.UNICODE
)

def tokenize(text):
    return token.findall(text)

In [81]:
df["input_tokens"] = df["Code Switched"].apply(tokenize)
df["target_tokens"] = df["Arabic"].apply(tokenize)

In [86]:
df["input_tokens"][19999]

['الصرع',
 'يعد',
 'من',
 'الـ',
 'diseases',
 'العصبية',
 'التي',
 'تصيب',
 'الكبار',
 'والصغار',
 '،',
 'ونوبات',
 'الصرع',
 'الشديدة',
 'تستعصي',
 'على',
 'الـ',
 'treatment',
 'ما',
 'يزيد',
 'من',
 'ألم',
 'ومعاناة',
 'المصابين',
 '.']

## Vocabulary

In [95]:
from tensorflow.keras.preprocessing.text import Tokenizer

input_tokenizer = Tokenizer(filters='', lower=False, oov_token='[oov]')
target_tokenizer = Tokenizer(filters='', lower=False, oov_token='[oov]')

input_tokenizer.fit_on_texts(df['input_tokens'])
target_tokenizer.fit_on_texts(df['target_tokens'])

print("Input vocabulary:", len(input_tokenizer.word_index) + 1)
print("Target vocabulary:", len(target_tokenizer.word_index) + 1)

Input vocabulary: 48894
Target vocabulary: 48959
